# 02 - Preprocessing & Underwater Degradation Pipeline

In real underwater super-resolution scenarios, HR ground truth is degraded by:
1. **Spatial downsampling** (optical resolution limits / sensor distance)
2. **Turbidity & forward scattering blur** (particulate suspension)
3. **Sensor noise** (amplified under low-light marine conditions)

This notebook demonstrates and visualizes the degradation models used to prepare LR/HR training pairs.

In [ ]:
import sys
sys.path.append("../src")

import torch
import torchvision.transforms.functional as TF
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import numpy as np

# Load sample image
img_path = "../sample_images/underwater_sample.png"
hr_img = Image.open(img_path).convert("RGB")
print(f"Loaded HR image: {hr_img.size}")

### Degradation Function: 2x and 4x Downscaling + Underwater Blur

In [ ]:
def apply_underwater_degradation(img: Image.Image, scale: int = 4, blur_radius: float = 1.0) -> Image.Image:
    # 1. Turbidity / scattering blur
    blurred = img.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    # 2. Downsampling to LR
    lr_w = max(1, img.width // scale)
    lr_h = max(1, img.height // scale)
    lr = blurred.resize((lr_w, lr_h), Image.Resampling.BICUBIC)
    return lr

lr_x2 = apply_underwater_degradation(hr_img, scale=2)
lr_x4 = apply_underwater_degradation(hr_img, scale=4)

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(hr_img)
axs[0].set_title(f"HR Ground Truth ({hr_img.width}x{hr_img.height})")
axs[0].axis("off")

axs[1].imshow(lr_x2)
axs[1].set_title(f"Degraded LR 2x ({lr_x2.width}x{lr_x2.height})")
axs[1].axis("off")

axs[2].imshow(lr_x4)
axs[2].set_title(f"Degraded LR 4x ({lr_x4.width}x{lr_x4.height})")
axs[2].axis("off")
plt.tight_layout()
plt.show()

### Verifying Dataset Loading with UnderwaterSRDataset

In [ ]:
from underwater_sr.dataset import UnderwaterSRDataset

# Test dataset with sample directory
ds = UnderwaterSRDataset(hr_dir="../sample_images", scale_factor=4, patch_size=64, is_train=True)
print(f"Dataset size: {len(ds)}")
if len(ds) > 0:
    lr_t, hr_t = ds[0]
    print(f"LR Patch Tensor shape: {lr_t.shape} | HR Patch Tensor shape: {hr_t.shape}")